In [7]:

import os 
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from dotenv import load_dotenv

In [10]:

client = MultiServerMCPClient({
    "webz_search": { 
        "transport": "http",
        "url" : "https://news-search-mcp.webz.io/mcp",
        "headers": {
            "Authorization": f"Bearer {os.environ.get('WEBZ_API_TOKEN')}"
        }
    }
})


async def main():
    async with client.session("webz_search") as session:
        tools = await load_mcp_tools(session)
        return tools

tools = await main()

# tools

In [11]:
load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

tools = await client.get_tools()
agent = create_agent(
    llm, 
    tools = tools,
)

result = await agent.ainvoke({"messages": "What is the latest news about AI?"})

result

{'messages': [HumanMessage(content='What is the latest news about AI?', additional_kwargs={}, response_metadata={}, id='7fae5d11-0c07-44e5-89be-ef24fb6f418f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What is the latest news about AI?" Need to fetch recent news about AI. Use news_search_by_webz. Probably want top recent articles, maybe last few days. Use query "artificial intelligence" or "AI". Return maybe 5-10. Use language English, sort by date_desc. Let\'s fetch.', 'tool_calls': [{'id': 'fc_6c25098f-7e2f-45dc-92e9-c0e1db5d6fb6', 'function': {'arguments': '{"days":3,"k":10,"language":["english"],"min_similarity":0.4,"query":"artificial intelligence OR AI","score_gte":4,"sort_by":"date_desc"}', 'name': 'news_search_by_webz'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 861, 'total_tokens': 1013, 'completion_time': 0.563820126, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_tim